# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected!")

Connected!


In [2]:
con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## Feature Vector

The feature vector contains historical measurements that are available before making a CTR or engagement improvement decision.

For this lane, I selected search visibility and engagement metrics that describe each content page without including future information.

In [3]:
feature_vector = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,

    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    scroll_events,
    sessions_ai

FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE

LIMIT 20;
""")

feature_vector


┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬──────────────────┬───────────────┬─────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ scroll_events │ sessions_ai │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      int64       │     int64     │    int64    │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼──────────────────┼───────────────┼─────────────┤
│ client_65de48885f4ef01b │ content_5c80451459c29b4a │ 2026-03-01  │               5 │          0 │               27 │             0 │           0 │
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │ 2026-03-01  │              39 │          0 │              221 │             0 │           1 │
│ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │ 2026-03-01  │             179 │          0 │       

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

| Feature | Meaning | Missing Value Handling | Available Before Prediction? |
|---------|---------|------------------------|------------------------------|
| gsc_impressions | Number of times the page appeared in Google Search results. | Rows without available GSC data are filtered using gsc_data_available IS TRUE. | Yes |
| gsc_clicks | Number of clicks the page received from Google Search. | Rows without available GSC data are filtered. | Yes |
| gsc_sum_position | Sum of Google Search positions used to describe search visibility. | Rows without available GSC data are filtered. | Yes |
| scroll_events | Number of scroll interactions recorded for the page. | Rows without available GA4 data are filtered using ga4_data_available IS TRUE. | Yes |
| sessions_ai | Number of AI-related sessions recorded for the page. | Rows without available GA4 data are filtered. | Yes |

In [5]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
    SUM(CASE WHEN gsc_sum_position IS NULL THEN 1 ELSE 0 END) AS missing_position,
    SUM(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END) AS missing_scroll_events,
    SUM(CASE WHEN sessions_ai IS NULL THEN 1 ELSE 0 END) AS missing_sessions_ai

FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE;
""")

┌────────────┬─────────────────────┬────────────────┬──────────────────┬───────────────────────┬─────────────────────┐
│ total_rows │ missing_impressions │ missing_clicks │ missing_position │ missing_scroll_events │ missing_sessions_ai │
│   int64    │       int128        │     int128     │      int128      │        int128         │       int128        │
├────────────┼─────────────────────┼────────────────┼──────────────────┼───────────────────────┼─────────────────────┤
│     364347 │                   0 │              0 │                0 │                     0 │                   0 │
└────────────┴─────────────────────┴────────────────┴──────────────────┴───────────────────────┴─────────────────────┘

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Leakage Hunt

I checked each feature to make sure it is available before the prediction is made.

The selected features (gsc_impressions, gsc_clicks,gsc_sum_position, scroll_events, and sessions_ai) are historical measurements and do not contain future information.

To demonstrate feature leakage, I intentionally included leaked_ctr, which is directly derived from gsc_clicks and gsc_impressions. I trained a simple decision tree with and without this feature. The model achieved much higher accuracy when leaked_ctr was included because it contains direct information about the target. This demonstrates why label-derived features must be excluded from the final feature vector.

In [18]:
import pandas as pd

df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    scroll_events,
    sessions_ai,

    CASE
        WHEN gsc_impressions > 0
        THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
        ELSE NULL
    END AS leaked_ctr

FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
    AND gsc_impressions > 0
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_avg_position,scroll_events,sessions_ai,leaked_ctr
0,5,0,5.400000,0,0,0.000000
1,39,0,5.666667,0,1,0.000000
2,179,0,5.156425,0,1,0.000000
3,72,0,7.694444,0,1,0.000000
4,3282,1,6.167885,0,0,0.000305


In [19]:
median_ctr = df["leaked_ctr"].median()

df["target"] = (df["leaked_ctr"] > median_ctr).astype(int)

print(df["target"].value_counts())

target
0    182310
1    182037
Name: count, dtype: int64


In [20]:
#training without leakage
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

safe_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "scroll_events",
    "sessions_ai"
]

X = df[safe_features]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy without leakage:",
      accuracy_score(y_test, pred))

Accuracy without leakage: 0.6940030190750652


## **Testing to see how our data is affected with and without leakage**

In [22]:
#testing with leakage
leak_features = safe_features + ["leaked_ctr"]

X = df[leak_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy WITH leakage:",
      accuracy_score(y_test, pred))

Accuracy WITH leakage: 1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## What I Excluded and Why

| Excluded Field | Why it was excluded |
|----------------|---------------------|
| client_hash_id | An identifier used only to identify clients. It does not contain predictive information and could cause the model to memorize clients instead of learning patterns. |
| content_hash_id | A unique identifier for each content page. It is used for identification only and should not be a model feature. |
| report_date | Used as context to identify when the data was collected. It is not a predictive feature for this task. |
| gsc_data_available | Used only to filter rows with valid Google Search Console data. It is not a meaningful feature. |
| ga4_data_available | Used only to filter rows with valid Google Analytics data. It is not a meaningful feature. |
| client_has_gsc | Indicates whether a client has GSC connected. It reflects system configuration rather than page performance. |
| client_has_ga4 | Indicates whether a client has GA4 connected. It reflects system configuration rather than page performance. |
| leaked_ctr | Deliberately created for the leakage experiment. It is derived from clicks and impressions and directly represents the target, so it must not be used for training. |

In [17]:
import sklearn
print(sklearn.__version__)

1.6.1


In [23]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5;
""")

┌─────────────────────────┬──────────────────────────┬─────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │
│         varchar         │         varchar          │    date     │    boolean     │    boolean     │      boolean       │      boolean       │
├─────────────────────────┼──────────────────────────┼─────────────┼────────────────┼────────────────┼────────────────────┼────────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │ true           │ false          │ true               │ NULL               │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │ true           │ false          │ true               │ NULL               │
│ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │ 2026-03-01  │ true           │ false          │ true               │ NULL  

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.